# autoresearch-team — Kaggle run (GPU T4 x2)

Save & Run All. Before starting: add the secret `GEMINI_API_KEY`
(Add-ons → Secrets) and attach this project as a dataset (or clone it in
cell 1). Everything persists under `/kaggle/working`; reattach that output
as an input dataset in the next session to resume (state.json, E11).

In [ ]:
# 1. Locate the project and install dependencies.
import os, shutil, subprocess, sys
from pathlib import Path
SRC = Path('/kaggle/input/autoresearch-team')  # attached dataset
# OPTION B: clone instead:
# subprocess.run(['git','clone','<your-fork-url>','/kaggle/working/src-proj'], check=True); SRC = Path('/kaggle/working/src-proj')
PROJECT = Path('/kaggle/working/autoresearch-team')
if not PROJECT.exists():
    shutil.copytree(SRC, PROJECT)  # keep repo/.git! orchestrator needs worktrees
os.chdir(PROJECT)
# if the upload stripped repo/.git (some dataset uploads skip dot-dirs), re-init it:
if not (PROJECT / 'repo' / '.git').exists():
    for a in (['init','-q'], ['add','-A'], ['-c','user.email=k@kaggle','-c','user.name=k',
              'commit','-q','-m','t4-port baseline']):
        subprocess.run(['git','-C',str(PROJECT/'repo'),*a], check=True)
    print('repo/.git re-initialized')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',
                'torch==2.9.1','rustbpe>=0.1.0','tiktoken>=0.11.0',
                'pyarrow>=21.0.0','numpy>=2.2.6','pandas>=2.3.3'], check=True)
print(subprocess.run(['nvidia-smi','--list-gpus'],capture_output=True,text=True).stdout)

In [ ]:
# 2. Secrets -> env (A4). Key only in this process's env; the training
# subprocess gets a whitelisted env without it (G7, E10).
import os
from kaggle_secrets import UserSecretsClient
os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
print('secret loaded:', bool(os.environ.get('GEMINI_API_KEY')))

In [ ]:
# 3. One-time data prep + tokenizer (small-GPU config from the T4 port).
import subprocess, sys
subprocess.run([sys.executable,'prepare.py','--num-shards','4'], cwd='repo', check=True)

In [ ]:
# 4. Phase 1 measurement (A8): one baseline run on GPU 0.
# startup = total_seconds - training_seconds; set training.startup_allowance_s
# in config.yaml to ~2x that. UNVERIFIED until this cell runs.
import subprocess, sys, os
env = {k:v for k,v in os.environ.items() if k in ('PATH','HOME')}
env['CUDA_VISIBLE_DEVICES'] = '0'
p = subprocess.run([sys.executable,'train.py'], cwd='repo', env=env,
                   capture_output=True, text=True, timeout=1500)
print(p.stdout[-1500:]); print(p.stderr[-500:])

In [ ]:
# 5. A7 check: two concurrent runs (GPU 0 + GPU 1); compare spread vs
# the single run above. Full concurrent calibration = E13, done inside cell 6.
import subprocess, sys, os
def run_on(gpu):
    env = {k:v for k,v in os.environ.items() if k in ('PATH','HOME')}
    env['CUDA_VISIBLE_DEVICES'] = str(gpu)
    return subprocess.Popen([sys.executable,'train.py'], cwd='repo', env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
procs = [run_on(0), run_on(1)]
for gpu, p in enumerate(procs):
    out,_ = p.communicate(timeout=1500)
    print(f'--- gpu{gpu} ---'); print(out[-400:])

In [ ]:
# 6. Full A/B evaluation (Section 9.1). For a smoke test (E14) first set
# budget.max_rounds: 2 in config.yaml. Repeats per docs/COMPUTE_BUDGET.md.
import subprocess, sys
p = subprocess.run([sys.executable,'-m','evals.run_ab','--config','config.yaml',
                    '--repeats','2','--out','results'], text=True, capture_output=True)
print(p.stdout[-3000:]); print(p.stderr[-1000:])

In [ ]:
# 7. Report + plots.
from pathlib import Path
print(Path('results/ab_report.md').read_text())
from IPython.display import Image
Image('results/val_bpb_curves.png')

In [ ]:
# 8. Persist. Resume later: attach this output as an input dataset, copy
# memory/ + work/ back into the project; orchestrator resumes from state.json.
import shutil
shutil.make_archive('/kaggle/working/autoresearch_results','zip','/kaggle/working')
print('saved /kaggle/working/autoresearch_results.zip')